# 🔬 Science Paper Analyzer
## Colab Notebook — Анализ научных статей на достоверность

Программа собирает статьи по вашей теме из 10+ научных источников и анализирует их по трём критериям:
1. **Цитируемость** — количество ссылок на статью
2. **Текст** — поиск признаков AI-генерации (через эвристики)
3. **Журнал** — проверка по списку хищнических журналов Beall's List

**Тема диссертации**: Разработка методов фильтрации генеративных текстовых данных для предотвращения коллапса языковых моделей

## 1️⃣ Установка зависимостей

In [ ]:
# Установка необходимых библиотек
import sys
!pip install -q streamlit requests pandas openpyxl lxml beautifulsoup4 numpy scikit-learn plotly
print("✅ Зависимости установлены")

## 2️⃣ Клонирование репозитория (или загрузка кода)

In [ ]:
# Загрузка кода из репозитория (замените URL на ваш)
# !git clone https://github.com/ВАШ_АККАУНТ/science-paper-analyzer.git
# %cd science-paper-analyzer

# Если вы загружаете файлы вручную через Colab:
from google.colab import files
import os, zipfile

# Раскомментируйте для загрузки архива с кодом:
# uploaded = files.upload()
# for fn in uploaded.keys():
#     with zipfile.ZipFile(fn, 'r') as z:
#         z.extractall('.')

print("📂 Код готов к работе")

## 3️⃣ Импорт модулей и запуск анализа

In [ ]:
# Импорт системы
import sys
sys.path.insert(0, '.')

from analyzer import PaperAnalyzer
import pandas as pd

print("✅ Модули загружены")

## 4️⃣ Запуск анализа по вашей теме

In [ ]:
# === НАСТРОЙКА ===
# Введите тему для поиска (на английском для лучших результатов)
QUERY = "filtering generative text data language model collapse"
MAX_PER_SOURCE = 5  # статей с каждого источника

# === ЗАПУСК ===
print(f"🔍 Поиск статей по теме: '{QUERY}'")
analyzer = PaperAnalyzer()

# Сбор статей
papers = analyzer.collect_papers(QUERY, max_per_source=MAX_PER_SOURCE)
print(f"📚 Найдено статей: {len(papers)}")

# Анализ
df = analyzer.analyze_papers(papers)
print(f"\n✅ Анализ завершён!")

## 5️⃣ Результаты

In [ ]:
# Показать результаты
display_cols = ['title', 'authors', 'year', 'source', 'overall_score', 'verdict']
df_display = df[display_cols].copy()
df_display.columns = ['Название', 'Авторы', 'Год', 'Источник', 'Score', 'Вердикт']

display(df_display)

In [ ]:
# Статистика
real_count = len(df[df['verdict'] == 'Real ✅'])
sus_count = len(df[df['verdict'] == 'Suspicious ⚠️'])
fake_count = len(df[df['verdict'] == 'Fake ❌'])

print(f"""
═══════════════════════════════════════
         📊 СВОДКА РЕЗУЛЬТАТОВ
═══════════════════════════════════════
  Всего статей:     {len(df)}
  ✅ Достоверные:    {real_count} ({real_count/len(df)*100:.1f}%)
  ⚠️ Подозрительные: {sus_count} ({sus_count/len(df)*100:.1f}%)
  ❌ Фейковые:       {fake_count} ({fake_count/len(df)*100:.1f}%)
═══════════════════════════════════════""")

# Показать подозрительные/фейковые
if sus_count > 0 or fake_count > 0:
    print("\n🔍 Статьи, требующие внимания:")
    suspicious = df[df['verdict'] != 'Real ✅'][display_cols]
    display(suspicious)

## 6️⃣ Экспорт результатов

In [ ]:
# Экспорт в CSV
from google.colab import files

csv_path = 'analysis_results.csv'
analyzer.export_csv(df, csv_path)
print(f"📄 CSV сохранён: {csv_path}")

# Экспорт в Excel
xlsx_path = 'analysis_results.xlsx'
analyzer.export_excel(df, xlsx_path)
print(f"📊 Excel сохранён: {xlsx_path}")

# Скачать файлы
files.download(csv_path)
# files.download(xlsx_path)  # Раскомментируйте для скачивания Excel

## 🔧 Дополнительно: веб-интерфейс

Для запуска полноценного веб-приложения (Streamlit) выполните в терминале Colab:

In [ ]:
# Для запуска Streamlit в Colab используйте LocalTunnel
# !pip install -q streamlit
# !wget -q -O - ipv4.icanhazip.com
# !streamlit run app.py & npx localtunnel --port 8501

print("💡 Веб-интерфейс:")
print("  Для локального запуска: streamlit run app.py")
print("  Для Colab: см. код выше (закомментирован)")

## 📚 О системе

**Science Paper Analyzer** — система верификации научных статей, разработанная в рамках диссертационной работы.

### Источники данных:
- arXiv.org (API)
- Semantic Scholar (API)
- OpenReview.net (API)
- ACL Anthology (API)
- JMLR.org (scraping)
- ResearchGate (через CrossRef API)
- CyberLeninka.ru (scraping)
- eLibrary.ru (через CrossRef API)
- Dissercat.com (через CrossRef API)
- FIPS.ru (через CrossRef API)

### Критерии оценки:
1. 📊 **Цитируемость** — количество ссылок на статью
2. 📝 **Текст** — эвристический анализ на признаки AI-генерации
3. 🏛️ **Журнал** — проверка по Beall's List (1200+ хищнических журналов)

### Возможности:
- Поиск по ключевым словам
- Детальный разбор каждой статьи
- Экспорт в CSV и Excel с условным форматированием
- Веб-интерфейс (Streamlit)